# Analyzing Merge Decisions

**Key methods:**
- `get_cluster_distances()` - Distance matrix between clusters (PRIMARY METHOD)
- `get_merge_decisions_df()` - See which criteria failed
- Experiment with `merge_global_percentile`

In [ ]:
from face_cluster import (
    PipelineConfig, QualityGater, KNNGraphBuilder,
    ConnectedComponentsClusterer, D10ExemplarSelector,
    ConservativeMerger
)
from face_cluster.analysis import ClusterSnapshot
import numpy as np

# Your faces loaded here
# faces = load_faces(...)

## 1. Configure

In [ ]:
config = PipelineConfig(
    K=5,
    distance_threshold=0.35,
    blur_min=50.0,
    merge_enabled=True,
    merge_margin=0.0,
    merge_exemplar_percentile=90,  # Per-cluster (P90)
    merge_global_percentile=50,    # Global (50=median, 75=permissive)
    merge_threshold_alpha=0.7,
)

## 2. Run pipeline

In [ ]:
# Quality gating
gater = QualityGater(config)
faces = gater.compute_blur_scores(faces)
core_indices, holdout_indices = gater.select_core_set(faces)

# Graph
graph_builder = KNNGraphBuilder(config)
distance_matrix = graph_builder.build_distance_matrix(faces, core_indices)
graph_result = graph_builder.build_mutual_knn_graph(distance_matrix, config.K, config.distance_threshold)

# Cluster
clusterer = ConnectedComponentsClusterer(config)
initial_result = clusterer.cluster(graph_result, core_indices)

# Exemplars
exemplar_selector = D10ExemplarSelector(config)
initial_result = exemplar_selector.select_exemplars(initial_result, graph_result)

# Merge
merger = ConservativeMerger(config)
merged_result = merger.merge_clusters(initial_result, graph_result)

## 3. Create snapshot

In [ ]:
merged_snapshot = ClusterSnapshot.from_result(
    merged_result,
    faces,
    core_indices,
    distance_matrix,
    stage="after_merge",
    config=config,
    cluster_thresholds=merger.last_thresholds,
    merge_candidates=merger.last_candidates
)

merged_snapshot.print_summary()

## 4. PRIMARY: Get cluster-to-cluster distances

In [ ]:
# This shows WHY clusters didn't merge
df = merged_snapshot.get_cluster_distances()
print(df.head(20))

**Columns:**
- `Exemplar_Dist` - Used for merge proposal (must be ≤ 0.45)
- `Min_Dist` - Closest any two faces
- `Mean_Dist` - Average
- `Max_Dist` - Furthest

In [ ]:
# Why didn't (4, 23) merge?
row = df[(df['C1']==4) & (df['C2']==23)].iloc[0]
print(f"Exemplar_Dist: {row['Exemplar_Dist']}")
if row['Exemplar_Dist'] > 0.45:
    print("→ Not proposed (exemplar dist > threshold)")

## 5. Get pairwise distances between two clusters

In [ ]:
# All pairwise distances
nodes_a = merged_snapshot.clusters[4]
nodes_b = merged_snapshot.clusters[23]
pairwise = merged_snapshot.distance_matrix[np.ix_(nodes_a, nodes_b)]
print(f"Shape: {pairwise.shape}")
print(pairwise)

## 6. Get exemplar distances between two clusters

In [ ]:
# Exemplar distances
exemplars_a = merged_snapshot.exemplars[4]
exemplars_b = merged_snapshot.exemplars[23]
ex_dists = merged_snapshot.distance_matrix[np.ix_(exemplars_a, exemplars_b)]
print(f"Shape: {ex_dists.shape}")
print(f"Min (used for merge): {ex_dists.min():.3f}")
print(ex_dists)

## 7. Experiment with global percentile

In [ ]:
for global_pct in [25, 50, 75, 90]:
    config.merge_global_percentile = global_pct
    merger = ConservativeMerger(config)
    test_result = merger.merge_clusters(initial_result, graph_result)
    
    print(f"P{global_pct}: {test_result.n_clusters} clusters")